<a href="https://colab.research.google.com/github/yawarabbasmalik/NLP-Text-Analysis-on-Yelp-Dataset/blob/main/NLP_Text_Analysis_on_Yelp_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Basic libraries

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import nltk
import random
import itertools
from collections import defaultdict

# Preprocessing

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from itertools import combinations
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import gensim
from keras.preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from imblearn.under_sampling import NearMiss, RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN

# Models

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegressionCV
import lightgbm as lgb

# Evaluation

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix, make_scorer
from lime import lime_text
from sklearn.pipeline import make_pipeline
from lime.lime_text import LimeTextExplainer


import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
# Loading Comments Dataset & Skipping Bad Rows Data.

data = pd.read_csv('/content/Yelp Dataset2.csv', encoding='utf-8', engine = 'python', error_bad_lines=False)
data.head()

,User_id,Product_id,Rating,Date,Review,Label
0,923,0,3,12/8/2014,The food at snack is a selection of popular Gr...,1
1,924,0,3,5/16/2013,This little place in Soho is wonderful. I had ...,1
2,925,0,4,7/1/2013,ordered lunch for 15 from Snack last Friday. Â...,1
3,926,0,4,7/28/2011,This is a beautiful quaint little restaurant o...,1
4,927,0,4,11/1/2010,Snack is great place for a Â casual sit down l...,1


In [ ]:
data = data.drop(['User_id'], axis=1)
data = data.drop(['Product_id'], axis=1)
data = data.drop(['Rating'], axis=1)
data = data.drop(['Date'], axis=1)
data.head()

,Review,Label
0,The food at snack is a selection of popular Gr...,1
1,This little place in Soho is wonderful. I had ...,1
2,ordered lunch for 15 from Snack last Friday. Â...,1
3,This is a beautiful quaint little restaurant o...,1
4,Snack is great place for a Â casual sit down l...,1


In [ ]:
# Remove null values from the DataFrame
data.dropna(subset=['Label', 'Review'], inplace=True)

In [ ]:
print(data['Label'].isnull().sum())
data['Review'].isnull().sum()

0


0

In [ ]:
# Confirming the Length of each comment/review and adding the same in the dataframe

data['review_length'] = data['Review'].apply(lambda x: len(x.split(' ')))
data.head()

,Review,Label,review_length
0,The food at snack is a selection of popular Gr...,1,40
1,This little place in Soho is wonderful. I had ...,1,52
2,ordered lunch for 15 from Snack last Friday. Â...,1,32
3,This is a beautiful quaint little restaurant o...,1,90
4,Snack is great place for a Â casual sit down l...,1,102


In [ ]:
from keras.preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences
from keras.utils import to_categorical

# Create a list of tokens for each sentence
tokenizer = RegexpTokenizer(r'\w+')
data["tokens"] = data["Review"].apply(tokenizer.tokenize)

all_words = [word for tokens in data["tokens"] for word in tokens]
sentence_lengths = [len(tokens) for tokens in data["tokens"]]
VOCAB = sorted(list(set(all_words)))
print("%s words total, with a vocabulary size of %s" % (len(all_words), len(VOCAB)))
print("Max sentence length is %s" % max(sentence_lengths))

1163 words total, with a vocabulary size of 496
Max sentence length is 181


In [ ]:
# Loading the Gensim Framework for Word2Vec
import gensim.downloader as api
import gensim
word2vec = api.load('word2vec-google-news-300')

**Text Preprocessing**

In [ ]:
# Text preparation

def basic_preprocessing(df):

    df_temp = df.copy(deep = True)

    df_temp = df_temp.rename(index = str, columns = {'Review': 'text'})

    df_temp.loc[:, 'text'] = [text_prepare(x) for x in df_temp['text'].values]

    le = LabelEncoder()
    le.fit(df_temp['Label'])
    df_temp.loc[:, 'class_label'] = le.transform(df_temp['Label'])

    tokenizer = RegexpTokenizer(r'\w+')

    df_temp["tokens"] = df_temp["text"].apply(tokenizer.tokenize)

    return df_temp

In [ ]:
def text_prepare(text):

    REPLACE_BY_SPACE_RE = re.compile('[/(){}\[\]\|@,;]')
    BAD_SYMBOLS_RE = re.compile('[^0-9a-z #+_]')
    STOPWORDS = set(stopwords.words('english'))

    text = text.lower()
    text = REPLACE_BY_SPACE_RE.sub('', text) # replace REPLACE_BY_SPACE_RE symbols by space in text
    text = BAD_SYMBOLS_RE.sub('', text) # delete symbols which are in BAD_SYMBOLS_RE from text
    words = text.split()
    i = 0
    while i < len(words):
        if words[i] in STOPWORDS:
            words.pop(i)
        else:
            i += 1
    text = ' '.join(map(str, words))# delete stopwords from text

    return text

In [ ]:
def get_metrics(y_test, y_predicted):

    precision = precision_score(y_test, y_predicted, average='weighted')

    recall = recall_score(y_test, y_predicted, average='weighted')

    f1 = f1_score(y_test, y_predicted, average='weighted')

    accuracy = accuracy_score(y_test, y_predicted)
    return accuracy, precision, recall, f1


**BAG of WORDS (BOW)**

- The Bag-of-Words model represents text as a collection of unique words and their frequencies in a document or corpus.
- It discards the order and context of words and focuses solely on the occurrence of words.
- BoW treats each document as a set of words and creates a matrix representation where each row corresponds to a document, and each column represents a unique word in the corpus.
- This technique is simple and effective for text classification, information retrieval, and document similarity tasks.



In [ ]:
def BOW(data):

    df_temp = data.copy(deep = True)
    df_temp = basic_preprocessing(df_temp)

    count_vectorizer = CountVectorizer()
    count_vectorizer.fit(df_temp['text'])

    list_corpus = df_temp["text"].tolist()
    list_labels = df_temp["class_label"].tolist()

    X = count_vectorizer.transform(list_corpus)

    return X, list_labels

In [ ]:
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')

folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=40)

clf_lr = LogisticRegressionCV(cv=folds, solver='saga',
                              multi_class='multinomial', n_jobs=-1, random_state=40)
clf_dt = DecisionTreeClassifier(random_state=40)
clf_rf = RandomForestClassifier(random_state=40)
clf_nb = MultinomialNB()

df_res = pd.DataFrame(columns=['Model', 'Precision', 'Recall', 'F1-score', 'Accuracy'])

print()
print("Using BOW Technique")

# Bag of words approach
X, y = BOW(data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

# Logistic Regression
clf_lr.fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)
accuracy_lr, precision_lr, recall_lr, f1_lr = get_metrics(y_test, y_pred_lr)
df_res = df_res.append({'Model': 'LR',
                        'Precision': precision_lr,
                        'Recall': recall_lr,
                        'F1-score': f1_lr,
                        'Accuracy': accuracy_lr}, ignore_index=True)

# Decision Tree
clf_dt.fit(X_train, y_train)
y_pred_dt = clf_dt.predict(X_test)
accuracy_dt, precision_dt, recall_dt, f1_dt = get_metrics(y_test, y_pred_dt)
df_res = df_res.append({'Model': 'DT',
                        'Precision': precision_dt,
                        'Recall': recall_dt,
                        'F1-score': f1_dt,
                        'Accuracy': accuracy_dt}, ignore_index=True)

# Random Forest
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
accuracy_rf, precision_rf, recall_rf, f1_rf = get_metrics(y_test, y_pred_rf)
df_res = df_res.append({'Model': 'RF',
                        'Precision': precision_rf,
                        'Recall': recall_rf,
                        'F1-score': f1_rf,
                        'Accuracy': accuracy_rf}, ignore_index=True)


# Print the results
print()
print(df_res)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Using BOW Technique

  Model  Precision  Recall  F1-score  Accuracy
0    LR     0.5625    0.75  0.642857      0.75
1    DT     0.5000    0.50  0.500000      0.50
2    RF     0.5625    0.75  0.642857      0.75
3    NB     0.0625    0.25  0.100000      0.25


**Using TFIDF Technique**

### **TF-IDF (Term Frequency-Inverse Document Frequency):**
TF-IDF is a numerical statistic that reflects the importance of a word in a document within a collection or corpus. It is commonly used for text mining and information retrieval. TF-IDF combines two factors:

- Term Frequency (TF): Measures the frequency of a term in a document.
- Inverse Document Frequency (IDF): Measures the rarity of a term across all documents in the corpus.
- The TF-IDF value increases proportionally to the number of times a word appears in a document but is offset by the frequency of the word in the entire corpus.
- This technique helps to identify important words in a document relative to the entire corpus.

In [ ]:
def tfidf(data, ngrams = 1):

    df_temp = data.copy(deep = True)
    df_temp = basic_preprocessing(df_temp)

    tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, ngrams))
    tfidf_vectorizer.fit(df_temp['text'])

    list_corpus = df_temp["text"].tolist()
    list_labels = df_temp["class_label"].tolist()

    X = tfidf_vectorizer.transform(list_corpus)

    return X, list_labels

In [ ]:
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')

folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=40)

clf_lr = LogisticRegressionCV(cv=folds, solver='saga',
                              multi_class='multinomial', n_jobs=-1, random_state=40)
clf_dt = DecisionTreeClassifier(random_state=40)
clf_rf = RandomForestClassifier(random_state=40)
clf_nb = MultinomialNB()

df_res = pd.DataFrame(columns=['Model', 'Precision', 'Recall', 'F1-score', 'Accuracy'])

print()
print("Using TF IDF Technique")

# Bag of words approach
X, y = tfidf(data, ngrams = 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

# Logistic Regression
clf_lr.fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)
accuracy_lr, precision_lr, recall_lr, f1_lr = get_metrics(y_test, y_pred_lr)
df_res = df_res.append({'Model': 'LR',
                        'Precision': precision_lr,
                        'Recall': recall_lr,
                        'F1-score': f1_lr,
                        'Accuracy': accuracy_lr}, ignore_index=True)

# Decision Tree
clf_dt.fit(X_train, y_train)
y_pred_dt = clf_dt.predict(X_test)
accuracy_dt, precision_dt, recall_dt, f1_dt = get_metrics(y_test, y_pred_dt)
df_res = df_res.append({'Model': 'DT',
                        'Precision': precision_dt,
                        'Recall': recall_dt,
                        'F1-score': f1_dt,
                        'Accuracy': accuracy_dt}, ignore_index=True)

# Random Forest
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
accuracy_rf, precision_rf, recall_rf, f1_rf = get_metrics(y_test, y_pred_rf)
df_res = df_res.append({'Model': 'RF',
                        'Precision': precision_rf,
                        'Recall': recall_rf,
                        'F1-score': f1_rf,
                        'Accuracy': accuracy_rf}, ignore_index=True)

# Print the results
print()
print(df_res)



Using TF IDF Technique


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



  Model  Precision  Recall  F1-score  Accuracy
0    LR     0.5625    0.75  0.642857      0.75
1    DT     0.5000    0.50  0.500000      0.50
2    RF     0.5625    0.75  0.642857      0.75
3    NB     0.5625    0.75  0.642857      0.75


**Using Word2Vec**

- Word2Vec is a popular word embedding technique that represents words as dense vector representations.
- It captures the semantic relationships and meaning of words by mapping them into a continuous vector space.
- Word2Vec learns the word embeddings by training neural network models on large text corpora.
- The resulting word vectors capture syntactic and semantic relationships between words.
- Word2Vec is useful for various NLP tasks, including word similarity, word analogy, and text classification.





In [ ]:
def get_average_word2vec(tokens_list, vector, generate_missing=False, k=300):
    if len(tokens_list)<1:
        return np.zeros(k)
    if generate_missing:
        vectorized = [vector[word] if word in vector else np.random.rand(k) for word in tokens_list]
    else:
        vectorized = [vector[word] if word in vector else np.zeros(k) for word in tokens_list]
    length = len(vectorized)
    summed = np.sum(vectorized, axis=0)
    averaged = np.divide(summed, length)
    return averaged

def get_word2vec_embeddings(vectors, clean_questions, generate_missing=False):
    embeddings = clean_questions['tokens'].apply(lambda x: get_average_word2vec(x, vectors,
                                                                                generate_missing=generate_missing))
    return list(embeddings)

In [ ]:
def w2v(data):

    df_temp = data.copy(deep = True)
    df_temp = basic_preprocessing(df_temp)

    embeddings = get_word2vec_embeddings(word2vec, df_temp)
    list_labels = df_temp["class_label"].tolist()

    return embeddings, list_labels

In [ ]:
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')

folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=40)

clf_lr = LogisticRegressionCV(cv=folds, solver='saga',
                              multi_class='multinomial', n_jobs=-1, random_state=40)
clf_dt = DecisionTreeClassifier(random_state=40)
clf_rf = RandomForestClassifier(random_state=40)
clf_nb = MultinomialNB()

df_res = pd.DataFrame(columns=['Model', 'Precision', 'Recall', 'F1-score', 'Accuracy'])

print()
print("Using Word2Vec Technique")

# Bag of words approach
X, y = w2v(data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

# Logistic Regression
clf_lr.fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)
accuracy_lr, precision_lr, recall_lr, f1_lr = get_metrics(y_test, y_pred_lr)
df_res = df_res.append({'Model': 'LR',
                        'Precision': precision_lr,
                        'Recall': recall_lr,
                        'F1-score': f1_lr,
                        'Accuracy': accuracy_lr}, ignore_index=True)

# Decision Tree
clf_dt.fit(X_train, y_train)
y_pred_dt = clf_dt.predict(X_test)
accuracy_dt, precision_dt, recall_dt, f1_dt = get_metrics(y_test, y_pred_dt)
df_res = df_res.append({'Model': 'DT',
                        'Precision': precision_dt,
                        'Recall': recall_dt,
                        'F1-score': f1_dt,
                        'Accuracy': accuracy_dt}, ignore_index=True)

# Random Forest
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
accuracy_rf, precision_rf, recall_rf, f1_rf = get_metrics(y_test, y_pred_rf)
df_res = df_res.append({'Model': 'RF',
                        'Precision': precision_rf,
                        'Recall': recall_rf,
                        'F1-score': f1_rf,
                        'Accuracy': accuracy_rf}, ignore_index=True)


# Print the results
print()
print(df_res)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Using Word2Vec Technique

  Model  Precision  Recall  F1-score  Accuracy
0    LR     0.5625    0.75  0.642857      0.75
1    DT     0.3750    0.25  0.300000      0.25
2    RF     0.5625    0.75  0.642857      0.75


**Vector Sapce Model**

- The Vector Space Model (VSM) is a representation model that converts text documents into numerical vectors.
- It treats each document as a vector in a high-dimensional space, where each dimension represents a term or word from the vocabulary.
- VSM is based on the Bag-of-Words model and represents documents as a weighted combination of terms.
- The weights can be derived from techniques such as term frequency, inverse document frequency, or TF-IDF.
- VSM is widely used in information retrieval, document clustering, and text classification tasks.



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def VSM(data):
    df_temp = data.copy(deep=True)
    df_temp = basic_preprocessing(df_temp)

    count_vectorizer = CountVectorizer()
    count_vectorizer.fit(df_temp['text'])

    X = count_vectorizer.transform(df_temp['text'])
    y = df_temp['class_label']

    return X, y


In [ ]:
import nltk
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('stopwords')

folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=40)

clf_lr = LogisticRegressionCV(cv=folds, solver='saga',
                              multi_class='multinomial', n_jobs=-1, random_state=40)
clf_dt = DecisionTreeClassifier(random_state=40)
clf_rf = RandomForestClassifier(random_state=40)
clf_nb = MultinomialNB()

df_res = pd.DataFrame(columns=['Model', 'Precision', 'Recall', 'F1-score', 'Accuracy'])

print()
print("Using Vector Space Model Technique")

# Bag of words approach
X, y = VSM(data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=40)

# Logistic Regression
clf_lr.fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)
accuracy_lr, precision_lr, recall_lr, f1_lr = get_metrics(y_test, y_pred_lr)
df_res = df_res.append({'Model': 'LR',
                        'Precision': precision_lr,
                        'Recall': recall_lr,
                        'F1-score': f1_lr,
                        'Accuracy': accuracy_lr}, ignore_index=True)

# Decision Tree
clf_dt.fit(X_train, y_train)
y_pred_dt = clf_dt.predict(X_test)
accuracy_dt, precision_dt, recall_dt, f1_dt = get_metrics(y_test, y_pred_dt)
df_res = df_res.append({'Model': 'DT',
                        'Precision': precision_dt,
                        'Recall': recall_dt,
                        'F1-score': f1_dt,
                        'Accuracy': accuracy_dt}, ignore_index=True)

# Random Forest
clf_rf.fit(X_train, y_train)
y_pred_rf = clf_rf.predict(X_test)
accuracy_rf, precision_rf, recall_rf, f1_rf = get_metrics(y_test, y_pred_rf)
df_res = df_res.append({'Model': 'RF',
                        'Precision': precision_rf,
                        'Recall': recall_rf,
                        'F1-score': f1_rf,
                        'Accuracy': accuracy_rf}, ignore_index=True)


# Print the results
print()
print(df_res)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Using Vector Space Model Technique

  Model  Precision  Recall  F1-score  Accuracy
0    LR     0.5625    0.75  0.642857      0.75
1    DT     0.5000    0.50  0.500000      0.50
2    RF     0.5625    0.75  0.642857      0.75
